# Etapa 2 — Evaluación revisada (Colab)

**Versión limpia:** importar la evaluación anterior como **semilla**, revisarla con el corpus real, corregir lo necesario, correr diagnósticos y **congelar una única versión nueva**.

Este notebook no reejecuta Etapas 0–1 ni inventa respuestas jurídicas. Necesita `artifacts/full/documents.jsonl` de la Etapa 1, el repositorio del grupo y `etapa2_base_para_revisar.zip` (también acepta tu ZIP congelado anterior). Los tres conjuntos se mantienen separados de `eval/` y nunca se sobrescribe la versión previa.

**Orden:** 1) Entorno → 2) Importación → 3) Auditoría y planilla → 4) Correcciones → 5) Revisión final → 6) Congelamiento y exportación.

## 1. Entorno

Elegí GPU T4 si vas a seguir con otras etapas; **esta Etapa 2 no necesita una GPU**. El código utiliza el repositorio del grupo y monta Drive para preservar `artifacts/`. Si ya ejecutaste el bootstrap de los notebooks anteriores en este mismo runtime, conserva la carpeta y no vuelve a clonarla.

In [ ]:
from pathlib import Path
import os, sys, subprocess, hashlib, json

REPO_URL = "https://github.com/Jahdejosefina/projecto_ragTP2.git"
REPO = Path("/content/projecto_rag")
DRIVE_ROOT = Path("/content/drive/MyDrive/projecto_rag")

from google.colab import drive, files
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

if not (REPO / "environment.py").exists():
    # Para repositorio privado se admite el secreto GITHUB_TOKEN de Colab.
    command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO)]
    r = subprocess.run(command, capture_output=True, text=True)
    if r.returncode:
        from google.colab import userdata
        token = os.environ.get("GITHUB_TOKEN")
        if not token:
            try:
                token = userdata.get("GITHUB_TOKEN")
            except Exception:
                token = None
        if not token:
            raise RuntimeError("No se pudo clonar. Revisá acceso al repo o GITHUB_TOKEN.")
        authed_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
        r = subprocess.run(["git", "clone", "--depth", "1", authed_url, str(REPO)], capture_output=True)
        if r.returncode:
            raise RuntimeError("Falló la clonación autenticada. Revisá los permisos de lectura.")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DRIVE_ART = DRIVE_ROOT / "artifacts"
DRIVE_ART.mkdir(parents=True, exist_ok=True)
LOCAL_ART = REPO / "artifacts"
if LOCAL_ART.is_symlink():
    if LOCAL_ART.resolve() != DRIVE_ART.resolve():
        raise RuntimeError(f"artifacts/ apunta a otro Drive: {LOCAL_ART.resolve()}")
elif LOCAL_ART.exists():
    raise RuntimeError("artifacts/ es una carpeta real, no un enlace a Drive. No la sobreescribo.")
else:
    LOCAL_ART.symlink_to(DRIVE_ART, target_is_directory=True)
os.environ["LAB_USE_DRIVE"] = "1"

reqs = [REPO / n for n in ("requirements.txt", "requirements-group.txt") if (REPO / n).exists()]
stamp = Path("/content/.etapa2_pip_" + hashlib.sha256(b"".join(p.read_bytes() for p in reqs)).hexdigest()[:12])
if not stamp.exists():
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    for req in reqs:
        cmd += ["-r", str(req)]
    subprocess.run(cmd + ["bm25s", "PyStemmer"], check=True)
    stamp.touch()

import environment, schema
from schema import Document
ART = environment.artifacts_dir(False)
DOCUMENTOS = ART / "documents.jsonl"
assert DOCUMENTOS.is_file(), f"Falta {DOCUMENTOS}. Ejecutá antes 01_documents."
with DOCUMENTOS.open(encoding="utf-8") as f:
    docs = [Document.model_validate(json.loads(line)) for line in f if line.strip()]
print("Corpus:", len(docs), "documentos /", sum(len(d.sections) for d in docs), "secciones")
print("Ruta persistente:", ART)

Corpus: 170 documentos / 7070 secciones
Ruta persistente: /content/projecto_rag/artifacts/full


## 2. Importar una semilla editable

La primera ejecución solicita un ZIP. Subí **`etapa2_base_para_revisar.zip`**; también acepta `etapa2_congelada(1).zip` porque lee solo los tres `.jsonl` e ignora `FROZEN.json`. Se crea `artifacts/full/eval_review_v2/`. En futuras ejecuciones retoma el borrador sin volver a importar ni sobrescribir cambios.

In [ ]:
import io, zipfile, shutil
from collections import Counter

WORK = ART / "eval_review_v2"   # Cambiá el nombre si querés iniciar OTRA revisión desde cero.
NAMES = ("single_turn.jsonl", "dialogues.jsonl", "scenarios.jsonl")

if all((WORK / n).exists() for n in NAMES):
    print("Retomando borrador existente:", WORK)
else:
    if WORK.exists() and any(WORK.iterdir()):
        raise RuntimeError(f"{WORK} está incompleta. Revisá su contenido antes de continuar.")
    subida = files.upload()
    if len(subida) != 1:
        raise ValueError("Subí exactamente un ZIP de Etapa 2.")
    with zipfile.ZipFile(io.BytesIO(next(iter(subida.values())))) as z:
        extraido = {}
        for n in NAMES:
            matches = [m for m in z.namelist() if m == n or m.endswith('/' + n)]
            if len(matches) != 1:
                raise ValueError(f"ZIP ambiguo o incompleto: {n} ({matches})")
            data = z.read(matches[0]).decode("utf-8")
            for linea in data.splitlines():
                if linea.strip(): json.loads(linea)
            extraido[n] = data
    WORK.mkdir(parents=True, exist_ok=True)
    for n, data in extraido.items():
        (WORK / n).write_text(data, encoding="utf-8")

if (WORK / "FROZEN.json").exists():
    raise RuntimeError("Este WORK ya está congelado. Creá otra carpeta para revisar más.")

def leer(nombre):
    return [json.loads(x) for x in (WORK / nombre).read_text(encoding="utf-8").splitlines() if x.strip()]

def guardar(nombre, filas):
    if (WORK / "FROZEN.json").exists():
        raise RuntimeError("No se edita un conjunto congelado.")
    with (WORK / nombre).open("w", encoding="utf-8") as f:
        for item in filas:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

items, dialogos, escenarios = (leer(n) for n in NAMES)
print("Preguntas:", len(items), dict(Counter(i['category'] for i in items)))
print("Dev/Test:", dict(Counter(i['split'] for i in items)))
print("Diálogos:", len(dialogos), "turnos:", sum(len(d['turns']) for d in dialogos))
print("Escenarios:", len(escenarios), "sondas:", sum(len(s['probes']) for s in escenarios))
schema.validate_eval(WORK, DOCUMENTOS).show().raise_if_failed()

Retomando borrador existente: /content/projecto_rag/artifacts/full/eval_review_v2
Preguntas: 30 {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
Dev/Test: {'dev': 7, 'test': 23}
Diálogos: 8 turnos: 32
Escenarios: 3 sondas: 27
eval set (/content/projecto_rag/artifacts/full/eval_review_v2): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12


## 3. Generar la auditoría y la planilla de revisión

Se generan tres informes legibles con preguntas y evidencia, y **una planilla CSV con una fila por pregunta, turno y sonda**. Una celda `estado` empieza en `pendiente`; marcala `ok` o `corregido` **después** de verificar la evidencia (idealmente en Google Sheets, exportando otra vez a CSV). El sistema invalida automáticamente los estados cuando detecta que cambió el contenido de una fila.

No es suficiente que `schema.validate_eval()` dé OK: esa función verifica el esquema, **no** si una respuesta jurídica es verdadera ni si los gold alternativos son equivalentes.

In [ ]:
import csv, json, re, unicodedata
from IPython.display import display
import pandas as pd

DOCS = {d.doc_id: d for d in docs}
SEC = {(d.doc_id, s.section_id): d.text[s.start:s.end]
       for d in docs for s in d.sections}


def evidencias(golds, max_chars=2500):
    textos = []
    for g in golds:
        clave = (g["doc_id"], g["section_id"])
        texto = SEC.get(clave, "[REFERENCIA INVÁLIDA]")
        textos.append(f"[{clave[0]} / {clave[1]}]\n{texto[:max_chars]}"
                      + ("\n[...truncado...]" if len(texto) > max_chars else ""))
    return "\n\n".join(textos) if textos else "(sin gold: verificar abstención)"


def entradas_revision():
    resultados = []
    def agregar(tipo, id_, contenido, nombre, clase, gold, claves, veredicto, texto):
        payload = json.dumps(contenido, ensure_ascii=False, sort_keys=True)
        firma = hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]
        resultados.append(dict(tipo=tipo, id=id_, huella=firma, grupo=nombre,
                              categoria=clase, texto=texto, gold=json.dumps(gold, ensure_ascii=False),
                              answer_key=json.dumps(claves, ensure_ascii=False),
                              expected_verdict=veredicto or "", estado="pendiente", observaciones=""))
    for p in leer("single_turn.jsonl"):
        agregar("pregunta", p["item_id"], p, p["split"], p["category"], p["gold"],
                p.get("answer_key"), p.get("expected_verdict"), p["question"])
    for d in leer("dialogues.jsonl"):
        for t in d["turns"]:
            agregar("turno", f"{d['dialogue_id']}:t{t['turn']}", t, d["split"], t["depends_on"],
                    t["gold"], t.get("answer_key"), t.get("expected_verdict"), t["user"])
    for s in leer("scenarios.jsonl"):
        for p in s["probes"]:
            agregar("sonda", f"{s['scenario_id']}:{p['probe_id']}", p, s.get("split", "test"),
                    p["ability"], p.get("gold", []), p.get("answer_key"),
                    p.get("expected_verdict"), p["question"])
    return resultados


def generar_revision():
    planilla = WORK / "revision_manual.csv"
    anteriores = {}
    if planilla.exists():
        with planilla.open(newline="", encoding="utf-8-sig") as f:
            anteriores = {r["id"]: r for r in csv.DictReader(f)}
    filas = entradas_revision()
    for f in filas:
        viejo = anteriores.get(f["id"])
        if viejo and viejo["huella"] == f["huella"]:
            f["estado"] = viejo.get("estado", "pendiente")
            f["observaciones"] = viejo.get("observaciones", "")
    with planilla.open("w", newline="", encoding="utf-8-sig") as f:
        escritor = csv.DictWriter(f, fieldnames=list(filas[0]))
        escritor.writeheader()
        escritor.writerows(filas)

    items, dialogos, escenarios = (leer(n) for n in NAMES)
    lineas = ["# Preguntas y referencias — auditoría humana", ""]
    for p in items:
        lineas += [f"## {p['item_id']} [{p['category']} · {p['split']}]",
                   p["question"], f"**Claves:** {p.get('answer_key')}",
                   f"**Veredicto:** {p.get('expected_verdict')}",
                   "**Evidencia (leer en la norma):**", "```", evidencias(p["gold"]), "```", ""]
    (WORK / "auditoria_preguntas.md").write_text("\n".join(lineas), encoding="utf-8")

    lineas = ["# Guiones de conversación", ""]
    for d in dialogos:
        lineas += [f"## {d['dialogue_id']} [{d['split']}]", ""]
        for t in d["turns"]:
            lineas += [f"### Turno {t['turn']} · {t['depends_on']}",
                       f"**Usuario:** {t['user']}", f"**Respuesta guionada:** {t['assistant']}",
                       f"**Oráculo:** {t['oracle_rewrite']}",
                       f"**Evidencia:**\n```\n{evidencias(t['gold'], 1300)}\n```", ""]
    (WORK / "auditoria_dialogos.md").write_text("\n".join(lineas), encoding="utf-8")

    lineas = ["# Escenarios de memoria", ""]
    for s in escenarios:
        lineas += [f"## {s['scenario_id']} — {s['persona']}", ""]
        for ses in s["sessions"]:
            lineas += [f"### Sesión {ses['session_id']} ({ses['date']})"]
            for t in ses["turns"]:
                lineas += [f"- **Usuario:** {t['user']}", f"- **Asistente:** {t['assistant']}"]
        for p in s["probes"]:
            lineas += [f"#### {p['probe_id']} · {p['ability']} (tras {p['after_session']})",
                       p["question"], f"**Evidencia sesiones:** {p['evidence_sessions']}",
                       f"**Clave/veredicto:** {p.get('answer_key')} / {p.get('expected_verdict')}",
                       f"**Gold corpus, si aplica:** {p.get('gold')}", ""]
    (WORK / "auditoria_escenarios.md").write_text("\n".join(lineas), encoding="utf-8")

    pendientes = [f['id'] for f in filas if f['estado'] not in ('ok','corregido')]
    print(f"Planilla: {planilla} | {len(filas)-len(pendientes)}/{len(filas)} revisadas")
    print("Informes: auditoria_preguntas.md, auditoria_dialogos.md, auditoria_escenarios.md")
    return filas

filas_revision = generar_revision()
display(pd.DataFrame(filas_revision)[['tipo','id','categoria','texto','estado']].head(12))

Planilla: /content/projecto_rag/artifacts/full/eval_review_v2/revision_manual.csv | 0/89 revisadas
Informes: auditoria_preguntas.md, auditoria_dialogos.md, auditoria_escenarios.md


,tipo,id,categoria,texto,estado
0,pregunta,q001,topical,¿Puede un proveedor cobrarle al consumidor por...,pendiente
1,pregunta,q002,topical,"Según la Ley 24.240 del corpus, ¿cuánto dura l...",pendiente
2,pregunta,q003,topical,"Según la Ley 24.240 del corpus, ¿la constancia...",pendiente
3,pregunta,q004,topical,¿Las características prometidas en una publici...,pendiente
4,pregunta,q005,topical,¿En qué idioma debe redactarse el documento de...,pendiente
5,pregunta,q006,topical,¿Qué sucede con el plazo de la garantía legal ...,pendiente
6,pregunta,q007,topical,¿Cómo debe documentarse la garantía de un serv...,pendiente
7,pregunta,q008,topical,Si se interrumpe un servicio público domicilia...,pendiente
8,pregunta,q009,topical,Si un consumidor ejerce su derecho de revocaci...,pendiente
9,pregunta,q010,topical,¿Qué tipo de responsabilidad tienen el fabrica...,pendiente


## 4. Diagnóstico previo con BM25

El diagnóstico **no es un resultado final de la Etapa 3**: sirve para detectar preguntas demasiado literales, referencias gold dudosas y diferencias entre consultas de identificador y preguntas conceptuales. No cambies preguntas *solamente* para subir el puntaje. La evidencia jurídica debe decidir qué gold es válido.

In [ ]:
import bm25s, Stemmer

stem = Stemmer.Stemmer("spanish").stemWord
STOP = set("de la que el en y los se del las un por con no una su para es al lo como más a o sus le les".split())


def tokens_es(txt):
    txt = unicodedata.normalize("NFC", txt.lower())
    txt = re.sub(r"\b(\d{2})\.(\d{3})\b", r"\1\2", txt)
    return [stem(w) for w in re.findall(r"\w+", txt) if w not in STOP]


def diagnostico_bm25():
    entradas = list(SEC.items())
    contenido = [" ".join(tokens_es(doc_id + " " + txt)) for (doc_id, _), txt in entradas]
    buscador = bm25s.BM25()
    buscador.index(bm25s.tokenize(contenido, stopwords=[], show_progress=False))
    preguntas = leer("single_turn.jsonl")
    consultas = bm25s.tokenize([" ".join(tokens_es(p["question"])) for p in preguntas],
                              stopwords=[], show_progress=False)
    indices, _ = buscador.retrieve(consultas, k=5, show_progress=False)

    filas = []
    for i,p in enumerate(preguntas):
        gold = {(g['doc_id'],g['section_id']) for g in p['gold']}
        top5 = [entradas[int(j)][0] for j in indices[i]]
        pos = next((j for j,k in enumerate(top5,1) if k in gold), None)
        consulta = tokens_es(p['question'])
        gold_tokens = [tokens_es(SEC[k]) for k in gold if k in SEC]
        compartido5 = any(set(zip(*[t[x:] for x in range(5)])) & set(zip(*[consulta[x:] for x in range(5)]))
                          for t in gold_tokens if len(t)>=5 and len(consulta)>=5)
        filas.append(dict(id=p['item_id'], categoria=p['category'], split=p['split'],
                          gold_top5=pos, posible_copia=(compartido5 and p['category']!='identifier'),
                          primera_seccion=top5[0][1]))
    df = pd.DataFrame(filas)
    df.to_csv(WORK / 'diagnostico_bm25.csv', index=False)
    (WORK / 'diagnostico_bm25_config.json').write_text(json.dumps({
        'uso':'solo_diagnostico_precongelamiento', 'unidad':'seccion',
        'prefijo':'doc_id', 'stemmer':'spanish', 'k':5,
        'advertencia':'No utilizar como resultados de Etapa 3.'}, indent=2),encoding='utf-8')
    print('Alertas de copia:', int(df['posible_copia'].sum()))
    print('Gold en top5 (entre respondibles):', int(df['gold_top5'].notna().sum()))
    display(df)
    return df

diagnostico = diagnostico_bm25()

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/7070 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/7070 [00:00<?, ?it/s]

Alertas de copia: 0
Gold en top5 (entre respondibles): 25


,id,categoria,split,gold_top5,posible_copia,primera_seccion
0,q001,topical,dev,1.0,False,LEY-27250#ART-1
1,q002,topical,dev,1.0,False,LEY-26361#ART-9
2,q003,topical,dev,1.0,False,LEY-24240#ART-15
3,q004,topical,test,NaN,False,LEY-26994#ANEXO-I/ART-1526
4,q005,topical,test,1.0,False,LEY-26361#ART-7
5,q006,topical,test,1.0,False,LEY-24240#ART-16
6,q007,topical,test,1.0,False,LEY-24240#ART-24
7,q008,topical,test,1.0,False,LEY-24240#ART-30
8,q009,topical,test,1.0,False,LEY-26361#ART-14
9,q010,topical,test,1.0,False,LEY-24999#ART-4


In [ ]:
for qid in ["q004", "q024"]:
    pregunta = next(
        p for p in leer("single_turn.jsonl")
        if p["item_id"] == qid
    )

    print("\n" + "=" * 70)
    print("PREGUNTA:", qid)
    print(pregunta["question"])
    print("RESPUESTA ESPERADA:", pregunta["answer_key"])

    for g in pregunta["gold"]:
        clave = (g["doc_id"], g["section_id"])

        print("\nGOLD:", clave)
        if clave in SEC:
            print(SEC[clave][:2000])
        else:
            print("ERROR: sección gold no encontrada")


PREGUNTA: q004
¿Las características prometidas en una publicidad obligan al oferente aunque no se repitan en el contrato?
RESPUESTA ESPERADA: ['obligan', 'obliga', 'vinculante', 'vinculantes']

GOLD: ('LEY-24240', 'LEY-24240#ART-8')
**ARTICULO 8º** — Efectos de la

publicidad. Las precisiones formuladas en la publicidad o en anuncios,

prospectos, circulares u otros medios de difusión se tienen por

incluidas en el contrato con el consumidor y obligan al oferente.

En los casos en que las ofertas de bienes y servicios se realicen

mediante el sistema de compras telefónicas, por catálogos o por

correos, publicados por cualquier medio de comunicación, deberá figurar

el nombre, domicilio y número de CUIT del oferente.

*(Artículo

sustituido por punto 3.2 del Anexo II de la [Ley

N° 26.994](http://servicios.infoleg.gob.ar/infolegInternet/verNorma.do?id=235975) B.O. 08/10/2014 Suplemento. Vigencia: 1° de agosto de

2015, texto según art. 1° de la [Ley

N° 27.077](http://servicios.infole

## 5. Correcciones explícitas (OPCIONAL)

Usá los informes de `WORK` para decidir cambios. Editá los diccionarios de abajo **solo** en los casos que hayas contrastado con los documentos. Las claves del esquema son las mismas de `schema.EvalItem`, `schema.DialogueTurn` y `schema.Probe`. No hay parches ocultos al final del notebook. Si cambiás una pregunta utilizada como oráculo en un diálogo, también tenés que ajustar ese turno y verificar el split.

In [ ]:
# EJEMPLO (descomentá y editá solamente después de comprobar evidencia):
# CAMBIOS_PREGUNTAS = {"q001": {"question": "...", "gold": [{"doc_id":"LEY-24240", "section_id":"LEY-24240#ART-4"}]}}
CAMBIOS_PREGUNTAS = {
    "q001": {
        "question": (
            "¿Puede un proveedor cobrarle al consumidor "
            "por informarle las características de un producto?"
        ),
        "answer_key": [
            "gratuita", "gratuito",
            "sin costo", "sin cargo",
            "no puede cobrar", "no debe cobrar"
        ]
    },
    "q003": {
        "answer_key": [
            "piezas reemplazadas",
            "piezas reparadas",
            "piezas cambiadas"
        ]
    },
    "q004": {
        "question": (
            "¿Las características prometidas en una publicidad "
            "obligan al oferente aunque no se repitan en el contrato?"
        )
    },
    "q005": {
        "question": (
            "¿En qué idioma debe redactarse "
            "el documento de venta al consumidor?"
        )
    }
}
CAMBIOS_PREGUNTAS.update({
    "q006": {
        "question": (
            "¿Qué sucede con el plazo de la garantía legal mientras "
            "un producto está siendo reparado y el consumidor no puede usarlo?"
        ),
        "answer_key": [
            "prolongación", "prolonga", "se prolonga",
            "extiende", "se extiende", "amplía",
            "se suma al plazo", "se agrega al plazo"
        ]
    },

    "q007": {
        "question": (
            "¿Cómo debe documentarse la garantía "
            "de un servicio contratado?"
        )
    },

    "q008": {
        "question": (
            "Si se interrumpe un servicio público domiciliario, "
            "¿cuántos días tiene la empresa desde el reclamo "
            "para demostrar que no fue responsable?"
        )
    },

    "q009": {
        "question": (
            "Si un consumidor ejerce su derecho de revocación "
            "en una compra comprendida por la Ley 24.240, "
            "¿quién debe pagar los gastos de devolución?"
        )
    },

    "q010": {
        "question": (
            "¿Qué tipo de responsabilidad tienen el fabricante, "
            "el distribuidor y el vendedor por los daños causados "
            "por un producto defectuoso?"
        )
    }
})
CAMBIOS_PREGUNTAS.update({
    "q011": {
        "question": (
            "¿Qué advertencia sobre los riesgos para la salud "
            "debe aparecer en los envases de videojuegos?"
        )
    },

    "q012": {
        "question": (
            "¿A partir de qué edad está destinada la población "
            "contemplada por el sistema nacional de talles de indumentaria?"
        )
    },

    "q013": {
        "question": (
            "¿Cada cuánto tiempo debe realizarse el estudio "
            "antropométrico para actualizar el sistema de talles?"
        )
    },

    "q014": {
        "question": (
            "¿Cómo se denominan los viajes escolares que incluyen "
            "actividades formativas integradas al programa educativo?"
        )
    },

    "q015": {
        "question": (
            "Según el texto de la Ley 25.649 de nuestra base, "
            "¿con qué denominación debe identificarse "
            "un medicamento en una receta médica?"
        )
    }
})
CAMBIOS_PREGUNTAS.update({
    "q019": {
        "question": (
            "¿La Ley 24.806 sobre publicidad de enseñanza privada "
            "fue promulgada totalmente o parcialmente?"
        ),
        "answer_key": [
            "parcialmente",
            "promulgada parcialmente"
        ]
    },

    "q020": {
        "question": (
            "¿En qué fecha fue promulgada parcialmente "
            "la Ley 25.599 sobre turismo estudiantil?"
        ),
        "answer_key": [
            "13 de junio de 2002",
            "junio 13 de 2002",
            "13/06/2002",
            "2002-06-13"
        ]
    }
})
CAMBIOS_PREGUNTAS.update({
    "q021": {
        "question": (
            "¿Cuál es el número de la ley argentina que establece "
            "requisitos para la publicidad de cursos ofrecidos "
            "por instituciones de enseñanza privada?"
        ),
        "answer_key": ["24.806", "24806", "Ley 24.806"],
        "gold": [
            {
                "doc_id": "LEY-24806",
                "section_id": "LEY-24806#intro"
            }
        ]
    },
    "q022": {
        "question": (
            "¿Qué número de ley identifica a las entidades financieras "
            "alcanzadas por la obligación de respetar la elección "
            "de profesionales al firmar escrituras traslativas de dominio?"
        ),
        "answer_key": ["21.526", "21526", "Ley 21.526"]
    },
    "q023": {
        "question": (
            "¿Qué secretaría figura como autoridad de aplicación "
            "de la normativa sobre advertencias y calificaciones "
            "en envases de videojuegos?"
        )
    },
    "q024": {
        "question": (
            "¿Qué organismo nacional debe promover la actuación "
            "de la justicia en defensa de la legalidad "
            "y los intereses generales de la sociedad?"
        ),
        "answer_key": [
            "Ministerio Público Fiscal de la Nación",
            "Ministerio Público Fiscal"
        ]
    }
})
CAMBIOS_PREGUNTAS["q025"] = {
    "question": (
        "¿Qué actividad fue declarada de interés deportivo, "
        "cultural y sociorrecreativo en todo el territorio argentino?"
    )
}
CAMBIOS_TURNOS = {}  # Ejemplo: {("d001", 2): {"oracle_rewrite": "..."}}
CAMBIOS_SONDAS = {}  # Ejemplo: {("m001", "p1"): {"answer_key": ["..."]}}


def aplicar_cambios():
    if not any((CAMBIOS_PREGUNTAS, CAMBIOS_TURNOS, CAMBIOS_SONDAS)):
        print('Sin cambios explícitos: se conserva la semilla para revisar.')
        return
    preguntas, dialogos, escenarios = (leer(n) for n in NAMES)
    encontrados = set()
    for p in preguntas:
        if p['item_id'] in CAMBIOS_PREGUNTAS:
            p.update(CAMBIOS_PREGUNTAS[p['item_id']]); encontrados.add(p['item_id'])
    assert encontrados == set(CAMBIOS_PREGUNTAS), 'ID de pregunta inexistente.'
    encontrados = set()
    for d in dialogos:
        for t in d['turns']:
            k = (d['dialogue_id'], t['turn'])
            if k in CAMBIOS_TURNOS:
                t.update(CAMBIOS_TURNOS[k]); encontrados.add(k)
    assert encontrados == set(CAMBIOS_TURNOS), 'Turno inexistente.'
    encontrados = set()
    for s in escenarios:
        for p in s['probes']:
            k = (s['scenario_id'], p['probe_id'])
            if k in CAMBIOS_SONDAS:
                p.update(CAMBIOS_SONDAS[k]); encontrados.add(k)
    assert encontrados == set(CAMBIOS_SONDAS), 'Sonda inexistente.'

    # Validar TODO antes de escribir un solo archivo: transacción en memoria.
    from schema import EvalItem, Dialogue, Scenario
    [EvalItem.model_validate(x) for x in preguntas]
    [Dialogue.model_validate(x) for x in dialogos]
    [Scenario.model_validate(x) for x in escenarios]
    for n,datos in zip(NAMES,(preguntas,dialogos,escenarios)):
        guardar(n,datos)
    with (WORK / 'decision_log_stage2.md').open('a',encoding='utf-8') as f:
        f.write('\n## Cambios de evaluación (completar motivo humano)\n')
        f.write('Preguntas: '+repr(list(CAMBIOS_PREGUNTAS))+'\n')
        f.write('Turnos: '+repr(list(CAMBIOS_TURNOS))+'\n')
        f.write('Sondas: '+repr(list(CAMBIOS_SONDAS))+'\n')
        f.write('Motivo y evidencia contrastada: **COMPLETAR ANTES DE CONGELAR**.\n')
    print('Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.')

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


In [ ]:
CAMBIOS_TURNOS.update({
    ("d001", 2): {
        "answer_key": [
            "piezas reemplazadas",
            "piezas reparadas",
            "piezas cambiadas"
        ]
    },

    ("d001", 3): {
        "answer_key": [
            "gratuita",
            "gratuito",
            "sin costo",
            "sin cargo",
            "no puede cobrar",
            "no debe cobrar"
        ]
    },

    ("d002", 1): {
        "user": (
            "Según la Ley 26.043, ¿qué advertencia sobre la salud "
            "debe aparecer en los envases de videojuegos?"
        )
    }
})

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


In [ ]:
CAMBIOS_TURNOS.update({
    ("d003", 1): {
        "user": (
            "¿A partir de qué edad se aplica el Sistema Único "
            "Normalizado de Talles de Indumentaria?"
        ),
        "oracle_rewrite": (
            "¿A partir de qué edad se aplica el Sistema Único "
            "Normalizado de Talles de Indumentaria?"
        )
    },

    ("d003", 3): {
        "user": (
            "Cambiando de tema: ¿qué advertencia sanitaria "
            "deben llevar los envases de videojuegos?"
        )
    },

    ("d004", 1): {
        "user": (
            "Según la Ley 25.649, ¿con qué denominación "
            "deben identificarse los medicamentos en las recetas?"
        ),
        "oracle_rewrite": (
            "Según la Ley 25.649, ¿con qué denominación "
            "deben identificarse los medicamentos en las recetas?"
        )
    },

    ("d004", 3): {
        "user": (
            "Pasemos al turismo estudiantil: ¿cómo se denominan "
            "los viajes escolares que incluyen actividades "
            "del programa educativo?"
        )
    }
})

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


In [ ]:
CAMBIOS_TURNOS.update({
    ("d005", 1): {
        "user": (
            "Si una institución educativa tiene reconocimiento oficial, "
            "¿qué dato sobre la aprobación de cada carrera o curso "
            "debe incluir en su publicidad?"
        ),
        "oracle_rewrite": (
            "Si una institución educativa tiene reconocimiento oficial, "
            "¿qué dato sobre la aprobación de cada carrera o curso "
            "debe incluir en su publicidad?"
        )
    },

    ("d006", 1): {
        "user": (
            "Si se interrumpe un servicio público domiciliario, "
            "¿cuántos días tiene la empresa desde mi reclamo "
            "para demostrar que no fue responsable?"
        ),
        "oracle_rewrite": (
            "Si se interrumpe un servicio público domiciliario, "
            "¿cuántos días tiene la empresa desde un reclamo "
            "para demostrar que no fue responsable?"
        )
    },

    ("d006", 3): {
        "oracle_rewrite": (
            "Si se interrumpe un servicio público domiciliario, "
            "¿cuántos días tiene la empresa desde un reclamo "
            "para demostrar que no fue responsable?"
        )
    }
})

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


In [ ]:
CAMBIOS_TURNOS.update({
    ("d008", 2): {
        "oracle_rewrite": (
            "¿Las características prometidas en una publicidad "
            "obligan al oferente aunque no se repitan en el contrato?"
        ),
        "gold": [
            {
                "doc_id": "LEY-24240",
                "section_id": "LEY-24240#ART-8"
            }
        ]
    }
})

aplicar_cambios()

# Verificar que el cambio quedó guardado
d008 = next(
    d for d in leer("dialogues.jsonl")
    if d["dialogue_id"] == "d008"
)

turno2 = d008["turns"][1]

print("Pregunta:", turno2["user"])
print("Oracle rewrite:", turno2["oracle_rewrite"])
print("Gold:", turno2["gold"])

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.
Pregunta: En esa venta, ¿también obliga lo prometido en la publicidad aunque no se repita en el contrato?
Oracle rewrite: ¿Las características prometidas en una publicidad obligan al oferente aunque no se repitan en el contrato?
Gold: [{'doc_id': 'LEY-24240', 'section_id': 'LEY-24240#ART-8'}]


In [ ]:
CAMBIOS_SONDAS.update({
    ("m001", "p1"): {
        "question": (
            "¿En qué academia te dije que compré el curso?"
        )
    },

    ("m001", "p4"): {
        "evidence_sessions": ["s1", "s2"]
    },

    ("m001", "p9"): {
        "evidence_sessions": ["s1", "s2"]
    }
})

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


In [ ]:
CAMBIOS_SONDAS.update({
    ("m002", "p4"): {
        "evidence_sessions": ["s1", "s2"]
    },
    ("m002", "p9"): {
        "evidence_sessions": ["s1", "s2"]
    }
})

aplicar_cambios()

Cambios aplicados. Volvé a generar la planilla y revisar filas alteradas.


## 6. Control final y actualización de la planilla

**Pasos humanos importantes:** (1) Leé los tres `.md` de auditoría, (2) corregí las referencias o preguntas dudosas, (3) marcá cada fila como `ok` o `corregido` en `revision_manual.csv` de Drive, y (4) volvé a correr esta celda. Guardá el CSV en la **misma ruta** antes de ejecutarla; cualquier fila modificada volverá a `pendiente`.

In [ ]:
schema.validate_eval(WORK, DOCUMENTOS).show().raise_if_failed()

# Evitar que una pregunta idéntica en diálogo dev se evalúe también en test, o viceversa.
por_pregunta = {p['question']:p for p in leer('single_turn.jsonl')}
for d in leer('dialogues.jsonl'):
    for t in d['turns']:
        p = por_pregunta.get(t['oracle_rewrite'])
        if p and p['split'] != d['split']:
            raise ValueError(f"Fuga dev/test en {d['dialogue_id']} turno {t['turn']} ↔ {p['item_id']}")

filas_revision = generar_revision()
recuento = Counter(f['estado'].strip().lower() for f in filas_revision)
print('Estados de revisión:',dict(recuento))
diagnostico = diagnostico_bm25()
print('Si cambiaste contenido, revisá nuevamente gold, claves, test/dev y los flags BM25.')

eval set (/content/projecto_rag/artifacts/full/eval_review_v2): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
Planilla: /content/projecto_rag/artifacts/full/eval_review_v2/revision_manual.csv | 0/89 revisadas
Informes: auditoria_preguntas.md, auditoria_dialogos.md, auditoria_escenarios.md
Estados de revisión: {'pendiente': 89}


DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/7070 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/7070 [00:00<?, ?it/s]

Alertas de copia: 0
Gold en top5 (entre respondibles): 25


,id,categoria,split,gold_top5,posible_copia,primera_seccion
0,q001,topical,dev,1.0,False,LEY-27250#ART-1
1,q002,topical,dev,1.0,False,LEY-26361#ART-9
2,q003,topical,dev,1.0,False,LEY-24240#ART-15
3,q004,topical,test,NaN,False,LEY-26994#ANEXO-I/ART-1526
4,q005,topical,test,1.0,False,LEY-26361#ART-7
5,q006,topical,test,1.0,False,LEY-24240#ART-16
6,q007,topical,test,1.0,False,LEY-24240#ART-24
7,q008,topical,test,1.0,False,LEY-24240#ART-30
8,q009,topical,test,1.0,False,LEY-26361#ART-14
9,q010,topical,test,1.0,False,LEY-24999#ART-4


Si cambiaste contenido, revisá nuevamente gold, claves, test/dev y los flags BM25.


In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[:5]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("\nEVIDENCIA:")
    print(evidencias(p["gold"], max_chars=2000))


ID: q001
Categoría: topical
Split: dev
Pregunta: ¿Puede un proveedor cobrarle al consumidor por informarle las características de un producto?
Respuesta esperada: ['gratuita', 'gratuito', 'sin costo', 'sin cargo', 'no puede cobrar', 'no debe cobrar']
Veredicto: no

EVIDENCIA:
[LEY-24240 / LEY-24240#ART-4]
**ARTICULO 4º** — Información. El proveedor está

obligado a suministrar al consumidor en forma cierta, clara y detallada

todo lo relacionado con las características esenciales de los bienes y

servicios que provee, y las condiciones de su comercialización.

La información debe ser siempre gratuita para el consumidor y

proporcionada en soporte físico, con claridad necesaria que permita su

comprensión. Solo se podrá suplantar la comunicación en soporte físico

si el consumidor o usuario optase de forma expresa por utilizar

cualquier otro medio alternativo de comunicación que el proveedor ponga

a disposición.

*(Artículo sustituido por art. 1° de

la [Ley

N° 27.250](http://servic

In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[5:10]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("\nEVIDENCIA:")
    print(evidencias(p["gold"], max_chars=1500))


ID: q006
Categoría: topical
Split: test
Pregunta: ¿Qué sucede con el plazo de la garantía legal mientras un producto está siendo reparado y el consumidor no puede usarlo?
Respuesta esperada: ['prolongación', 'prolonga', 'se prolonga', 'extiende', 'se extiende', 'amplía', 'se suma al plazo', 'se agrega al plazo']
Veredicto: None

EVIDENCIA:
[LEY-24240 / LEY-24240#ART-16]
**ARTICULO 16.** — Prolongación del Plazo

de Garantía. El tiempo durante el cual el consumidor está privado del

uso de la cosa en garantía, por cualquier causa relacionada con su

reparación, debe computarse como prolongación del plazo de garantía

legal.



ID: q007
Categoría: topical
Split: test
Pregunta: ¿Cómo debe documentarse la garantía de un servicio contratado?
Respuesta esperada: ['por escrito', 'escrita']
Veredicto: None

EVIDENCIA:
[LEY-24240 / LEY-24240#ART-24]
**ARTICULO 24.** — Garantía. La garantía sobre un

contrato de prestación de servicios deberá documentarse por escrito

haciendo constar:

a) La c

In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[10:15]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("\nEVIDENCIA:")
    print(evidencias(p["gold"], max_chars=1500))


ID: q011
Categoría: topical
Split: test
Pregunta: ¿Qué advertencia sobre los riesgos para la salud debe aparecer en los envases de videojuegos?
Respuesta esperada: ['la sobreexposición es perjudicial para la salud']
Veredicto: None

EVIDENCIA:
[LEY-26043 / LEY-26043#ART-1]
##### ARTICULO 1º — Los fabricantes y/o importadores de videojuegos deberán colocar en los envases en que comercialicen esos productos la leyenda: "La sobreexposición es perjudicial para la salud". Asimismo se deberá incluir la calificación "Apta para todo público", "Apta para mayores de 13 años" y "Apta para mayores de 18 años" según corresponda. En el caso de la exhibición y/o uso de videojuegos con acceso al público, se deberá exhibir la leyenda y la calificación antes del inicio del mismo.



[LEY-26043 / LEY-26043#intro]
# VIDEOJUEGOS

VIDEOJUEGOS

Ley 26.043

Establécese que los fabricantes y/o importadores de videojuegos deberán colocar en los envases en que comercialicen dichos productos la leyenda "La sobre

In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[15:20]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("\nEVIDENCIA:")
    print(evidencias(p["gold"], max_chars=1500))


ID: q016
Categoría: metadata
Split: dev
Pregunta: ¿En qué fecha fue sancionada la Ley 27.250, según su encabezado en el corpus?
Respuesta esperada: ['18 de mayo de 2016', 'mayo 18 de 2016', '18/05/2016', '2016-05-18']
Veredicto: None

EVIDENCIA:
[LEY-27250 / LEY-27250#intro]
# DEFENSA DEL CONSUMIDOR

**DEFENSA

DEL CONSUMIDOR**

**Ley 27250**

**Modificación. Ley N° 24.240.**

**Sancionada: Mayo 18 de 2016**

**Promulgada de Hecho: Junio 08 de 2016**

El Senado y Cámara de Diputados de la

Nación Argentina reunidos en Congreso, etc. sancionan con fuerza de

Ley:



ID: q017
Categoría: metadata
Split: test
Pregunta: ¿En qué fecha fue sancionada la ley de videojuegos 26.043, según su encabezado en el corpus?
Respuesta esperada: ['1 de junio de 2005', 'junio 1 de 2005', '01/06/2005', '1/6/2005', '2005-06-01']
Veredicto: None

EVIDENCIA:
[LEY-26043 / LEY-26043#intro]
# VIDEOJUEGOS

VIDEOJUEGOS

Ley 26.043

Establécese que los fabricantes y/o importadores de videojuegos deberán colocar en 

In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[20:25]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("\nEVIDENCIA:")
    print(evidencias(p["gold"], max_chars=1500))


ID: q021
Categoría: identifier
Split: dev
Pregunta: ¿Cuál es el número de la ley argentina que establece requisitos para la publicidad de cursos ofrecidos por instituciones de enseñanza privada?
Respuesta esperada: ['24.806', '24806', 'Ley 24.806']
Veredicto: None

EVIDENCIA:
[LEY-24806 / LEY-24806#intro]
# PUBLICIDAD DE LA ENSEÑANZA PRIVADA

PUBLICIDAD DE LA ENSEÑANZA PRIVADA

Ley 24.806

Requisitos que deberá cumplir la difusión de los servicios que presten las personas y/o instituciones de propiedad privada, destinada a enseñanza, que dicten cursos presenciales, semipresenciales o a distancia.

Sancionada: Abril 16 de 1997

Promulgada Parcialmente: Mayo 5 de 1997

El Senado y Cámara de Diputados de la Nación Argentina reunidos en Congreso, etc. sancionan con fuerza de Ley:



ID: q022
Categoría: identifier
Split: test
Pregunta: ¿Qué número de ley identifica a las entidades financieras alcanzadas por la obligación de respetar la elección de profesionales al firmar escrituras traslat

In [ ]:
print(SEC.get(
    ("LEY-27665", "LEY-27665#intro"),
    "No se encontró la sección"
)[:1200])

# ACTIVIDADES DE INTERES DEPORTIVO, CULTURAL Y SOCIORRECREATIVO

**ACTIVIDADES DE INTERÉS DEPORTIVO, CULTURAL Y SOCIORRECREATIVO**

**Ley 27665**

**Declaración.**

El Senado y Cámara de Diputados de la Nación Argentina reunidos en Congreso, etc. sancionan con fuerza de

Ley:




In [ ]:
preguntas = leer("single_turn.jsonl")

for p in preguntas[25:30]:
    print("\n" + "=" * 75)
    print("ID:", p["item_id"])
    print("Categoría:", p["category"])
    print("Split:", p["split"])
    print("Pregunta:", p["question"])
    print("Respondible:", p.get("answerable"))
    print("Respuesta esperada:", p.get("answer_key"))
    print("Veredicto:", p.get("expected_verdict"))
    print("Gold:", p["gold"])

    if p["gold"]:
        print("\nEVIDENCIA:")
        print(evidencias(p["gold"], max_chars=1500))


ID: q026
Categoría: out_of_corpus
Split: test
Pregunta: ¿Qué resolución se dictó en mi reclamo de defensa del consumidor con número de expediente PRUEBA-CONSUMO-X731?
Respondible: False
Respuesta esperada: None
Veredicto: None
Gold: []

ID: q027
Categoría: out_of_corpus
Split: test
Pregunta: ¿Cuántos meses de garantía comercial adicional prometió la tienda ficticia CompraDelta-X731 en el contrato privado de mi compra? Pregunto por la adicional, no por la garantía legal.
Respondible: False
Respuesta esperada: None
Veredicto: None
Gold: []

ID: q028
Categoría: out_of_corpus
Split: dev
Pregunta: ¿A qué hora tengo asignada mi audiencia de conciliación del reclamo ficticio PRUEBA-CONSUMO-X942?
Respondible: False
Respuesta esperada: None
Veredicto: None
Gold: []

ID: q029
Categoría: metadata
Split: test
Pregunta: Según el encabezado de la Ley 27.250 del corpus, ¿en qué fecha fue promulgada de hecho?
Respondible: True
Respuesta esperada: ['8 de junio de 2016', '08 de junio de 2016', 'junio 0

In [ ]:
import json

documentos = [
    json.loads(linea)
    for linea in open(ART / "documents.jsonl", encoding="utf-8")
    if linea.strip()
]

identificadores = [
    "PRUEBA-CONSUMO-X731",
    "CompraDelta-X731",
    "PRUEBA-CONSUMO-X942"
]

for identificador in identificadores:
    encontrados = [
        doc.get("doc_id", "sin ID")
        for doc in documentos
        if identificador.casefold()
        in json.dumps(doc, ensure_ascii=False).casefold()
    ]

    print(f"{identificador}: {len(encontrados)} documentos")
    if encontrados:
        print("Encontrados en:", encontrados)

PRUEBA-CONSUMO-X731: 0 documentos
CompraDelta-X731: 0 documentos
PRUEBA-CONSUMO-X942: 0 documentos


dialogos:

In [ ]:
import json

dialogos = leer("dialogues.jsonl")

for dialogo in dialogos[:2]:
    print("\n" + "=" * 80)
    print(json.dumps(
        dialogo,
        ensure_ascii=False,
        indent=2
    ))


{
  "dialogue_id": "d001",
  "turns": [
    {
      "turn": 1,
      "user": "Según la Ley 24.240 del corpus, ¿cuánto dura la garantía legal de un bien mueble usado no consumible, si no se acuerda un plazo mayor?",
      "assistant": "La garantía legal es de tres meses para esos bienes usados, salvo que se acuerde un plazo mayor.",
      "depends_on": "standalone",
      "oracle_rewrite": "Según la Ley 24.240 del corpus, ¿cuánto dura la garantía legal de un bien mueble usado no consumible, si no se acuerda un plazo mayor?",
      "gold": [
        {
          "doc_id": "LEY-24240",
          "section_id": "LEY-24240#ART-11"
        },
        {
          "doc_id": "LEY-26361",
          "section_id": "LEY-26361#ART-9"
        }
      ],
      "answerable": true,
      "answer_key": [
        "tres meses",
        "3 meses"
      ],
      "expected_verdict": null
    },
    {
      "turn": 2,
      "user": "Y si me lo reparan en garantía, ¿el comprobante debe decir qué piezas cambiaron

In [ ]:
import json

dialogos = leer("dialogues.jsonl")

for dialogo in dialogos[2:4]:
    print("\n" + "=" * 80)
    print(json.dumps(
        dialogo,
        ensure_ascii=False,
        indent=2
    ))


{
  "dialogue_id": "d003",
  "turns": [
    {
      "turn": 1,
      "user": "Según la Ley 27.521 del corpus, ¿a partir de qué edad se contempla a la población destinataria del sistema de talles?",
      "assistant": "La población destinataria se contempla a partir de los doce años.",
      "depends_on": "standalone",
      "oracle_rewrite": "Según la Ley 27.521 del corpus, ¿a partir de qué edad se contempla a la población destinataria del sistema de talles?",
      "gold": [
        {
          "doc_id": "LEY-27521",
          "section_id": "LEY-27521#ART-1"
        }
      ],
      "answerable": true,
      "answer_key": [
        "doce años",
        "12 años",
        "doce (12) años"
      ],
      "expected_verdict": null
    },
    {
      "turn": 2,
      "user": "¿Cada cuánto deben repetir el estudio corporal para actualizar ese sistema?",
      "assistant": "Cada diez años.",
      "depends_on": "coreference",
      "oracle_rewrite": "Según la Ley 27.521 del corpus, ¿cada cu

In [ ]:
import json

dialogos = leer("dialogues.jsonl")

for dialogo in dialogos[4:6]:
    print("\n" + "=" * 80)
    print(json.dumps(
        dialogo,
        ensure_ascii=False,
        indent=2
    ))


{
  "dialogue_id": "d005",
  "turns": [
    {
      "turn": 1,
      "user": "Según la Ley 24.806, cuando una institución educativa tiene reconocimiento oficial, ¿qué número relacionado con la aprobación de cada carrera o curso debe mencionar en su publicidad?",
      "assistant": "Debe mencionar el número de resolución por el que fue aprobada la carrera o el curso.",
      "depends_on": "standalone",
      "oracle_rewrite": "Según la Ley 24.806, cuando una institución educativa tiene reconocimiento oficial, ¿qué número relacionado con la aprobación de cada carrera o curso debe mencionar en su publicidad?",
      "gold": [
        {
          "doc_id": "LEY-24806",
          "section_id": "LEY-24806#ART-1"
        }
      ],
      "answerable": true,
      "answer_key": [
        "número de resolución",
        "numero de resolución",
        "resolución"
      ],
      "expected_verdict": null
    },
    {
      "turn": 2,
      "user": "¿Cuándo se sancionó esa ley?",
      "assistan

In [ ]:
import json

dialogos = leer("dialogues.jsonl")

for dialogo in dialogos[6:8]:
    print("\n" + "=" * 80)
    print(json.dumps(
        dialogo,
        ensure_ascii=False,
        indent=2
    ))


{
  "dialogue_id": "d007",
  "turns": [
    {
      "turn": 1,
      "user": "Según el encabezado de la Ley 27.250 del corpus, ¿en qué fecha fue promulgada de hecho?",
      "assistant": "La Ley 27.250 fue promulgada de hecho el 8 de junio de 2016.",
      "depends_on": "standalone",
      "oracle_rewrite": "Según el encabezado de la Ley 27.250 del corpus, ¿en qué fecha fue promulgada de hecho?",
      "gold": [
        {
          "doc_id": "LEY-27250",
          "section_id": "LEY-27250#intro"
        }
      ],
      "answerable": true,
      "answer_key": [
        "8 de junio de 2016",
        "08 de junio de 2016",
        "junio 08 de 2016",
        "08/06/2016",
        "2016-06-08"
      ],
      "expected_verdict": null
    },
    {
      "turn": 2,
      "user": "¿Y cuándo fue promulgada de hecho la Ley 26.043?",
      "assistant": "El 15 de julio de 2005.",
      "depends_on": "topic_shift",
      "oracle_rewrite": "Según el encabezado de la Ley 26.043 del corpus, ¿en qué 

memoria:

In [ ]:
import json

escenarios = leer("scenarios.jsonl")

print(json.dumps(
    escenarios[0],
    ensure_ascii=False,
    indent=2
))

{
  "scenario_id": "m001",
  "persona": "Clara, persona completamente ficticia que consulta documentación normativa sobre consumo.",
  "sessions": [
    {
      "session_id": "s1",
      "date": "2026-09-01T10:00:00",
      "turns": [
        {
          "user": "Soy Clara. Compré un curso de fotografía con material impreso enviado a domicilio en Academia Ejemplo Alfa y pagué 120000 pesos por el curso y el material. Mi canal preferido de contacto es el teléfono.",
          "assistant": "Registrado: compra en Academia Ejemplo Alfa, precio de 120000 pesos y contacto por teléfono."
        },
        {
          "user": "Para mis próximas consultas sobre normativa, usá el documento LEY-24806 de nuestro corpus. Su fecha de publicación registrada es 1997-05-13. Por ahora prefiero respuestas en un párrafo.",
          "assistant": "Tomaré como referencia LEY-24806, con fecha registrada 1997-05-13, y responderé en un párrafo."
        }
      ]
    },
    {
      "session_id": "s2",
      "d

In [ ]:
import json

escenarios = leer("scenarios.jsonl")

print(json.dumps(
    escenarios[1],
    ensure_ascii=False,
    indent=2
))

{
  "scenario_id": "m002",
  "persona": "Mateo, persona completamente ficticia que consulta documentación normativa sobre consumo.",
  "sessions": [
    {
      "session_id": "s1",
      "date": "2026-09-01T10:00:00",
      "turns": [
        {
          "user": "Soy Mateo. Compré un videojuego en formato físico en Tienda Ejemplo Beta y pagué 85000 pesos por el producto. Mi canal preferido de contacto es el teléfono.",
          "assistant": "Registrado: compra en Tienda Ejemplo Beta, precio de 85000 pesos y contacto por teléfono."
        },
        {
          "user": "Para mis próximas consultas sobre normativa, usá el documento LEY-26043 de nuestro corpus. Su fecha de publicación registrada es 2005-07-18. Por ahora prefiero respuestas en un párrafo.",
          "assistant": "Tomaré como referencia LEY-26043, con fecha registrada 2005-07-18, y responderé en un párrafo."
        }
      ]
    },
    {
      "session_id": "s2",
      "date": "2026-09-08T10:00:00",
      "turns": [
   

In [ ]:
import json

with open(ART / "documents.jsonl", encoding="utf-8") as f:
    documentos = [json.loads(line) for line in f if line.strip()]

doc = next(
    d for d in documentos
    if d.get("doc_id") == "LEY-26043"
)

print("Metadatos:", json.dumps(
    doc.get("metadata", {}),
    ensure_ascii=False,
    indent=2
))

Metadatos: {
  "date": "2005-07-18",
  "lang": "es",
  "source_status": "in_force",
  "source_url": "http://servicios.infoleg.gob.ar/infolegInternet/anexos/105000-109999/107891/norma.htm",
  "status": "unknown"
}


In [ ]:
import json

escenarios = leer("scenarios.jsonl")

print(json.dumps(
    escenarios[2],
    ensure_ascii=False,
    indent=2
))

{
  "scenario_id": "m003",
  "persona": "Inés, persona completamente ficticia que consulta documentación normativa sobre consumo.",
  "sessions": [
    {
      "session_id": "s1",
      "date": "2026-09-01T10:00:00",
      "turns": [
        {
          "user": "Soy Inés. Compré una campera en Indumentaria Ejemplo Gamma y pagué 64000 pesos por el producto. Mi canal preferido de contacto es el teléfono.",
          "assistant": "Registrado: compra en Indumentaria Ejemplo Gamma, precio de 64000 pesos y contacto por teléfono."
        },
        {
          "user": "Para mis próximas consultas sobre normativa, usá el documento LEY-27521 de nuestro corpus. Su fecha de publicación registrada es 2019-12-20. Por ahora prefiero respuestas en un párrafo.",
          "assistant": "Tomaré como referencia LEY-27521, con fecha registrada 2019-12-20, y responderé en un párrafo."
        }
      ]
    },
    {
      "session_id": "s2",
      "date": "2026-09-08T10:00:00",
      "turns": [
        {
 

In [ ]:
from copy import deepcopy
import json

# Cargar escenarios y seleccionar m003
todos = leer("scenarios.jsonl")
m003 = next(s for s in todos if s["scenario_id"] == "m003")
m003 = deepcopy(m003)

s1, s2, s3 = m003["sessions"]
assert [s["session_id"] for s in m003["sessions"]] == ["s1", "s2", "s3"]

# Verificar la fecha documental antes de conservarla
with open(DOCUMENTOS, encoding="utf-8") as f:
    documentos = [json.loads(x) for x in f if x.strip()]

doc = next(d for d in documentos if d["doc_id"] == "LEY-27521")
fecha = doc["metadata"].get("date")
print("Fecha registrada de LEY-27521:", fecha)

assert fecha == "2019-12-20", (
    "La fecha no coincide: revisar antes de continuar."
)

# SESIÓN 2: corregir el precio
s2["turns"][0] = {
    "user": (
        "Tengo que corregir un dato de mi compra: "
        "la campera costó 69000 pesos, no 64000. "
        "El precio que te había dicho antes era incorrecto."
    ),
    "assistant": (
        "Entendido. El precio correcto de la campera "
        "es 69000 pesos. Reemplazo el importe anterior."
    )
}

# SESIÓN 3: introducir el envío y mantener preferencia
s3["turns"][0] = {
    "user": (
        "Además de los 69000 pesos de la campera, "
        "pagué 2500 pesos de envío. No hubo otros gastos. "
        "A partir de ahora quiero respuestas con viñetas, "
        "no en un párrafo."
    ),
    "assistant": (
        "Registrado: 2500 pesos de envío, "
        "71500 pesos en total y respuestas con viñetas."
    )
}

probes = {p["probe_id"]: p for p in m003["probes"]}

# p2: combinación de las tres sesiones
probes["p2"].update({
    "after_session": "s3",
    "question": (
        "Considerando el precio corregido y el envío, "
        "¿cuánto terminé pagando por la compra?"
    ),
    "answer_key": ["71500", "71.500", "71 500"],
    "evidence_sessions": ["s1", "s2", "s3"],
    "notes": (
        "Revisión humana: requiere reemplazar el precio "
        "original y sumar el envío informado después."
    )
})

# p3: recuperar el dato histórico
probes["p3"].update({
    "question": (
        "¿Qué precio te había dicho inicialmente "
        "antes de corregirlo?"
    ),
    "answer_key": ["64000", "64.000", "64 000"],
    "evidence_sessions": ["s1", "s2"],
    "notes": "Revisión humana: recuperar el precio anterior."
})

# p4: recuperar el dato actualizado
probes["p4"].update({
    "question": "¿Cuál es el precio correcto de la campera ahora?",
    "answer_key": ["69000", "69.000", "69 000"],
    "evidence_sessions": ["s1", "s2"],
    "notes": (
        "Revisión humana: debe priorizar 69000 sobre "
        "el precio original de 64000."
    )
})

# p8 y p9: misma pregunta, distinta respuesta temporal
pregunta_control = "¿El precio de mi campera es de 64000 pesos?"

probes["p8"].update({
    "question": pregunta_control,
    "evidence_sessions": ["s1"],
    "expected_verdict": "yes",
    "notes": "Revisión humana: precio inicial, antes de corregirlo."
})

probes["p9"].update({
    "question": pregunta_control,
    "evidence_sessions": ["s1", "s2"],
    "expected_verdict": "no",
    "notes": "Revisión humana: el precio fue corregido a 69000."
})

# Validar antes de guardar
schema.Scenario.model_validate(m003)

for i, s in enumerate(todos):
    if s["scenario_id"] == "m003":
        todos[i] = m003

guardar("scenarios.jsonl", todos)

with (WORK / "decision_log_stage2.md").open(
    "a", encoding="utf-8"
) as f:
    f.write(
        "\n## Rediseño de m003\n"
        "Se incorporó una corrección explícita del precio, "
        "información nueva en una sesión posterior y "
        "sondas que distinguen el dato histórico del vigente. "
        "Motivo: reducir la redundancia entre los tres escenarios.\n"
    )

schema.validate_eval(WORK, DOCUMENTOS).show().raise_if_failed()
generar_revision()

print("\nm003 actualizado:")
for p in m003["probes"]:
    print(
        p["probe_id"],
        p["ability"],
        p["question"],
        p.get("answer_key"),
        p.get("expected_verdict")
    )

Fecha registrada de LEY-27521: 2019-12-20
eval set (/content/projecto_rag/artifacts/full/eval_review_v2): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
Planilla: /content/projecto_rag/artifacts/full/eval_review_v2/revision_manual.csv | 0/89 revisadas
Informes: auditoria_preguntas.md, auditoria_dialogos.md, auditoria_escenarios.md

m003 actualizado:
p1 extraction ¿En qué tienda te dije que hice la compra? ['Indumentaria Ejemplo Gamma'] None
p2 multi_session Considerando el precio corregido y el envío, ¿cuánto terminé pagando por la compra? ['71500', '71.500', '71 500'] None
p3 temporal ¿Qué precio te había dicho inicialmente antes de corregirlo? ['64000', '64.000', '

In [ ]:
import csv
from collections import Counter

with (WORK / "revision_manual.csv").open(
    encoding="utf-8-sig", newline=""
) as f:
    filas = list(csv.DictReader(f))

actuales = {x["id"]: x for x in entradas_revision()}

estados = Counter(f["estado"].strip().lower() for f in filas)

huellas_invalidas = [
    f["id"] for f in filas
    if f["id"] not in actuales
    or f["huella"] != actuales[f["id"]]["huella"]
]

print("Total de registros:", len(filas))
print("Estados:", dict(estados))
print("Huellas desactualizadas:", huellas_invalidas)

Total de registros: 89
Estados: {'pendiente': 89}
Huellas desactualizadas: []


In [ ]:
import csv
import shutil
from pathlib import Path
from collections import Counter
from google.colab import files

# Subir el CSV editado
subidos = files.upload()
nombre = next(iter(subidos))

if not nombre.lower().endswith(".csv"):
    raise ValueError("Tenés que subir un archivo CSV, no un PDF.")

nuevo = Path(nombre)
destino = WORK / "revision_manual.csv"

def cargar_csv(ruta):
    with ruta.open(encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

filas_nuevas = cargar_csv(nuevo)
filas_actuales = cargar_csv(destino)

actuales = {r["id"]: r for r in filas_actuales}
nuevas = {r["id"]: r for r in filas_nuevas}

assert len(filas_nuevas) == 89, "El CSV debe tener 89 filas"
assert set(nuevas) == set(actuales), "Los IDs no coinciden"

assert all(
    nuevas[id_]["huella"] == actuales[id_]["huella"]
    for id_ in actuales
), "Hay huellas diferentes"

assert all(
    r["estado"].strip().lower() in ("ok", "corregido")
    for r in filas_nuevas
), "Todavía existen estados pendientes o inválidos"

# Conservar copia anterior
shutil.copy2(destino, WORK / "revision_manual_backup.csv")

# Reemplazar con el CSV revisado
shutil.copy2(nuevo, destino)

print("CSV actualizado correctamente")
print("Total:", len(filas_nuevas))
print("Estados:", dict(Counter(
    r["estado"].strip().lower() for r in filas_nuevas
)))

Saving revision_manual - revision_manual.csv to revision_manual - revision_manual.csv
CSV actualizado correctamente
Total: 89
Estados: {'corregido': 70, 'ok': 19}


In [ ]:
log = WORK / "decision_log_stage2.md"

if log.exists():
    contenido = log.read_text(encoding="utf-8")
    pendientes = contenido.count("COMPLETAR ANTES DE CONGELAR")
    print("Justificaciones pendientes:", pendientes)
else:
    print("No se encontró el registro de decisiones.")

print("Congelamiento previo:", (WORK / "FROZEN.json").exists())

Justificaciones pendientes: 20
Congelamiento previo: False


In [ ]:
from pathlib import Path
import shutil

log = WORK / "decision_log_stage2.md"
texto = log.read_text(encoding="utf-8")

marcador = (
    "Motivo y evidencia contrastada: "
    "**COMPLETAR ANTES DE CONGELAR**."
)

cantidad = texto.count(marcador)

# Conservar el registro original
shutil.copy2(log, WORK / "decision_log_stage2.md.bak")

# Las entradas repetidas remiten a una justificación consolidada
texto = texto.replace(
    marcador,
    "Motivo y evidencia contrastada: "
    "ver el resumen consolidado de revisión al final "
    "del documento. Algunas aplicaciones corresponden "
    "a reejecuciones de correcciones anteriores."
)

texto += """

## Resumen consolidado de revisión de Etapa 2

Se inspeccionaron 30 preguntas individuales, 32 turnos de diálogo
y 27 sondas de memoria.

### Preguntas individuales
- Se mejoraron formulaciones para hacerlas más naturales.
- Se ampliaron claves de respuesta para reconocer variantes correctas.
- En q004 se eliminó la referencia gold demasiado amplia
  LEY-26994#ANEXO-II/ART-285, conservando LEY-24240#ART-8.
- Se diversificaron preguntas de metadatos e identificadores.
- Se comprobaron las referencias gold contra el corpus.
- Se verificó la ausencia de los identificadores ficticios
  utilizados en los tres casos out_of_corpus.

### Diálogos
- Se revisaron dependencias conversacionales, reformulaciones
  independientes, respuestas esperadas y referencias gold.
- Se sincronizaron claves de respuesta y mejoraron redacciones.
- En d008:t2 se eliminó la misma referencia gold amplia
  descartada en q004.

### Memoria
- Se revisaron las sesiones y las respuestas de las sondas.
- Se mejoraron anotaciones de evidencia de cambios de preferencias.
- Se rediseñó m003 para evaluar la corrección de un precio:
  64000 pesos iniciales, 69000 pesos corregidos,
  2500 pesos de envío y 71500 pesos de total.
- Se verificaron las respuestas temporales anteriores y posteriores
  a la corrección.

### Limitaciones
- Algunos diálogos reutilizan hechos de las preguntas individuales.
- m001 y m002 tienen estructuras similares.
- La validación automática comprueba la estructura,
  pero no reemplaza la revisión de las respuestas.
- El diagnóstico BM25 se utilizó para detectar casos dudosos,
  no para optimizar artificialmente el conjunto de evaluación.
"""

log.write_text(texto, encoding="utf-8")

print("Entradas documentadas:", cantidad)
print(
    "Justificaciones pendientes:",
    log.read_text(encoding="utf-8").count(
        "COMPLETAR ANTES DE CONGELAR"
    )
)

Entradas documentadas: 20
Justificaciones pendientes: 0


## 7. Congelar y descargar (solo cuando la revisión termine)

Marcá los 89 registros de `revision_manual.csv`, escribí tu nombre y las conclusiones de la revisión. El notebook rechaza los registros pendientes y cualquier inconsistencia estructural. Se genera `FROZEN.json` **una sola vez** en la carpeta nueva de Drive, se comprueba con `schema.load_eval` y se exporta un ZIP distinto del anterior. El notebook **no pisa** automáticamente el `eval/` del repositorio: eso requiere commit del grupo.

In [ ]:
from datetime import datetime, timezone

REVISOR = "Alexander Melnikov"

RESUMEN_REVISION = (
    "Se revisaron 30 preguntas individuales, 32 turnos "
    "y 27 sondas de memoria. Se verificaron las referencias "
    "gold, se corrigieron preguntas y se rediseñó m003."
)

CONFIRMO_REVISION = True

if not CONFIRMO_REVISION or not REVISOR.strip() or not RESUMEN_REVISION.strip():
    raise ValueError('Antes de congelar: completá REVISOR, RESUMEN_REVISION y CONFIRMO_REVISION=True.')
if (WORK / 'FROZEN.json').exists():
    raise RuntimeError('Este conjunto ya está congelado. No se reemplaza: creá otro WORK si hay cambios.')

schema.validate_eval(WORK, DOCUMENTOS).show().raise_if_failed()

# Verificación de firmas SIN reescribir la planilla (evita perder ediciones en Drive).
with (WORK / 'revision_manual.csv').open(newline='',encoding='utf-8-sig') as f:
    planilla = list(csv.DictReader(f))
actuales = entradas_revision()
originales = {x['id']:x for x in actuales}
if set(r['id'] for r in planilla) != set(originales) or len(planilla) != len(actuales):
    raise ValueError('La planilla no coincide con la evaluación actual (ids o cantidad distintos).')
pendientes = [r['id'] for r in planilla if r['huella'] != originales[r['id']]['huella']
              or r['estado'].strip().lower() not in ('ok','corregido')]
if pendientes:
    raise ValueError(f'Faltan {len(pendientes)} revisiones o cambió su contenido: {pendientes[:20]}')
log = WORK / 'decision_log_stage2.md'
if log.exists() and 'COMPLETAR ANTES DE CONGELAR' in log.read_text(encoding='utf-8'):
    raise ValueError('El decision_log_stage2.md contiene motivos sin completar.')

# Documentar revisión sin afirmar que la realizó un modelo.
registro = {
    'revisor':REVISOR.strip(), 'fecha_utc':datetime.now(timezone.utc).isoformat(),
    'descripcion':RESUMEN_REVISION.strip(),
    'registros_revision':len(planilla),
    'uso_ia':'Los ítems originales contienen notas de asistencia con IA; se revisaron contra el corpus.'
}
(WORK / 'revision_humana.json').write_text(json.dumps(registro,ensure_ascii=False,indent=2),encoding='utf-8')
with log.open('a',encoding='utf-8') as f:
    f.write(f"\n## Congelamiento {registro['fecha_utc']}\nRevisor: {REVISOR}\n{RESUMEN_REVISION}\n")

schema.freeze_eval(WORK, DOCUMENTOS)
schema.load_eval(WORK, documents_path=DOCUMENTOS)  # Validar huellas y corpus.

salida = ART / ('etapa2_congelada_v2_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '.zip')
with zipfile.ZipFile(salida,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in WORK.iterdir():
        if p.is_file() and p.suffix != '.bak':
            z.write(p,arcname='eval/'+p.name)
print('Congelado y verificado:',WORK)
print('ZIP nuevo:',salida)
files.download(str(salida))
print("Etapa 3: schema.load_eval(environment.artifacts_dir(False) / 'eval_review_v2', documents_path=environment.artifacts_dir(False) / 'documents.jsonl')")

eval set (/content/projecto_rag/artifacts/full/eval_review_v2): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
eval set (/content/projecto_rag/artifacts/full/eval_review_v2): OK
   single-turn items: 30
   by split: {'dev': 7, 'test': 23}
   by category: {'topical': 15, 'metadata': 7, 'identifier': 5, 'out_of_corpus': 3}
   dialogues: 8
   dialogue turns: 32
   scenarios: 3
   probes by ability: {'extraction': 3, 'multi_session': 6, 'temporal': 3, 'knowledge_update': 9, 'preference': 3, 'abstention': 3}
   yes/no rows (expected_verdict): 12
frozen: single_turn.jsonl (30 rows), dialogues.jsonl (8 rows), scenarios.jsonl (3 rows)


OSError: [Errno 95] Operation not supported: '/content/projecto_rag/artifacts/full/eval_review_v2/revision_manual.gsheet'

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

# Carpeta con la evaluación congelada
carpeta = Path(WORK)

# Verificar que el congelamiento existe
assert (carpeta / "FROZEN.json").is_file(), (
    "No se encontró FROZEN.json"
)

# Archivos obligatorios
obligatorios = [
    "single_turn.jsonl",
    "dialogues.jsonl",
    "scenarios.jsonl",
    "FROZEN.json",
    "revision_manual.csv",
    "decision_log_stage2.md",
]

# Informes adicionales (si existen)
opcionales = [
    "auditoria_preguntas.md",
    "auditoria_dialogos.md",
    "auditoria_escenarios.md",
]

faltantes = [
    nombre for nombre in obligatorios
    if not (carpeta / nombre).is_file()
]

if faltantes:
    raise FileNotFoundError(
        f"Faltan archivos: {faltantes}"
    )

# Crear un ZIP nuevo, sin archivos .gsheet
salida = carpeta.parent / "etapa2_congelada_v2_final.zip"

with ZipFile(salida, "w", ZIP_DEFLATED) as z:
    for nombre in obligatorios + opcionales:
        archivo = carpeta / nombre

        if archivo.is_file():
            z.write(archivo, arcname=f"eval/{nombre}")

# Verificar integridad
with ZipFile(salida) as z:
    assert z.testzip() is None
    print("Archivos incluidos:")
    for nombre in z.namelist():
        print(" -", nombre)

print("\nZIP generado correctamente:")
print(salida)

files.download(str(salida))

Archivos incluidos:
 - eval/single_turn.jsonl
 - eval/dialogues.jsonl
 - eval/scenarios.jsonl
 - eval/FROZEN.json
 - eval/revision_manual.csv
 - eval/decision_log_stage2.md
 - eval/auditoria_preguntas.md
 - eval/auditoria_dialogos.md
 - eval/auditoria_escenarios.md

ZIP generado correctamente:
/content/projecto_rag/artifacts/full/etapa2_congelada_v2_final.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>